# NYC Multiple Dwelling Registrations — Analysis

## Section 1: Setup & Libraries

Import the required libraries for data manipulation, statistical analysis, and visualization.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import plotly.express as px

## Section 2: Data Loading & Initial Exploration

Load the merged registrations and contacts dataset.  
Inspect missing values and preview the raw structure before any cleaning.

In [2]:
df = pd.read_csv(r"C:\Users\Diogo Forte\OneDrive\DTU\4TH SEMESTER\Social data analysis\final_project\merged_registrations_contacts.csv")

In [3]:
print(df.isna().sum())

registrationid                  0
buildingid                      0
boro                            0
zip                             0
bin                           137
lastregistrationdate            0
registrationcontactid        2217
type                         2217
contactdescription           3322
corporationname          15882733
businessstreetname        4928049
businesshousenumber       4928153
dtype: int64


In [4]:
df = pd.DataFrame(df)
df.head(15)

,registrationid,buildingid,boro,zip,bin,lastregistrationdate,registrationcontactid,type,contactdescription,corporationname,businessstreetname,businesshousenumber
0,825850,271859,BROOKLYN,11234,3223251.0,2026-03-19T00:00:00.000,82585016.0,JointOwner,JOINT,NaN,56 STREET,787 EAST
1,825850,271859,BROOKLYN,11234,3223251.0,2026-03-19T00:00:00.000,82585015.0,JointOwner,JOINT,NaN,56 STREET,787 EAST
2,825850,271859,BROOKLYN,11234,3223251.0,2026-03-19T00:00:00.000,82585004.0,Agent,JOINT,NaN,EAST 56TH STREET,787
3,825850,271859,BROOKLYN,11234,3223251.0,2026-03-19T00:00:00.000,82585005.0,HeadOfficer,JOINT,NaN,EAST 56TH STREEt,787
4,825850,271859,BROOKLYN,11234,3223251.0,2026-03-19T00:00:00.000,82585006.0,Officer,JOINT,NaN,EAST 56TH STREET,787
5,131572,169,MANHATTAN,10003,1006381.0,2025-08-13T00:00:00.000,13157203.0,CorporateOwner,GEN.PART,GRAY ROCK EQUITIES LLC,MELISSA CT,207-21
6,131572,169,MANHATTAN,10003,1006381.0,2025-08-13T00:00:00.000,13157204.0,Agent,GEN.PART,REGAL PROPERTY MANAGEMENT INC.,WEST 25 STREET,18
7,131572,169,MANHATTAN,10003,1006381.0,2025-08-13T00:00:00.000,13157205.0,HeadOfficer,GEN.PART,NaN,1ST AVE,141
8,131572,169,MANHATTAN,10003,1006381.0,2025-08-13T00:00:00.000,13157206.0,Officer,GEN.PART,NaN,1ST AVE,141
9,109808,245,MANHATTAN,10028,1050399.0,2025-08-08T00:00:00.000,10980803.0,CorporateOwner,CORP,"1576.FIRST AVENUE, INC",E 82 ST,400


## Section 3: Data Cleaning

Steps taken to clean the dataset:
- **Drop** low-value or overly null columns: `contactdescription`, `lastregistrationdate`, `registrationcontactid`, `bin`
- **Filter** to rows with a known `corporationname` — keeping only corporate entities

In [5]:
df= df.drop(columns=["contactdescription","lastregistrationdate","registrationcontactid","bin"])
df.dropna(subset=["corporationname"], inplace=True)

In [6]:
df = pd.DataFrame(df)
df.head(15)

,registrationid,buildingid,boro,zip,type,corporationname,businessstreetname,businesshousenumber
5,131572,169,MANHATTAN,10003,CorporateOwner,GRAY ROCK EQUITIES LLC,MELISSA CT,207-21
6,131572,169,MANHATTAN,10003,Agent,REGAL PROPERTY MANAGEMENT INC.,WEST 25 STREET,18
9,109808,245,MANHATTAN,10028,CorporateOwner,"1576.FIRST AVENUE, INC",E 82 ST,400
10,109808,245,MANHATTAN,10028,Agent,1576 FIRST AVE INC,E 82 ST,400
17,420463,810843,QUEENS,11434,CorporateOwner,Rochdale Village Inc.,137th Avenue,169-65
18,420463,810843,QUEENS,11434,Agent,Douglas Elliman Property Management,137th AVENUE,169-65
23,335058,383549,BROOKLYN,11231,CorporateOwner,220 UNION LLC,UNION STREET,220
25,125763,1043,MANHATTAN,10128,CorporateOwner,EL-KAM REALTY CO,THIRD AVENUE,777
26,125763,1043,MANHATTAN,10128,Agent,ROSE PROPERTY MANAGEMENT GROUP LLC,THIRD AVENUE,777
28,137997,1699,MANHATTAN,10021,CorporateOwner,179 EAST 70TH STREET CORP,THIRD AVE,909


## Section 4: Dataset Overview

Inspect the size and composition of the cleaned dataset:
- Total number of rows
- Unique buildings (`buildingid`)
- Contact types present (`CorporateOwner`, `Agent`, `Lessee`)
- Unique corporation names

In [7]:
print(f"Total rows: {len(df)}")

Total rows: 12011236


In [8]:
print(f"Number of building id_s: {len(df['buildingid'].unique())}")

Number of building id_s: 119513


In [9]:
type_list=list(df["type"].unique())
print(f"Number of types: {len(type_list)}")
print(f"Types' names: {type_list}")

Number of types: 3
Types' names: ['CorporateOwner', 'Agent', 'Lessee']


In [10]:
corporation_list = list(df["corporationname"].unique())
print(f"Number of corporations: {len(corporation_list)}")

Number of corporations: 103384


In [11]:
pd.Series(corporation_list).to_frame('corporationname')

,corporationname
0,GRAY ROCK EQUITIES LLC
1,REGAL PROPERTY MANAGEMENT INC.
2,"1576.FIRST AVENUE, INC"
3,1576 FIRST AVE INC
4,Rochdale Village Inc.
...,...
103379,EASTCHESTER 52 LLC
103380,171 HALSEY ST LLC
103381,JDM 2023 LLC
103382,355 PROPERTIES CORP


In [12]:
dupes = df[df.duplicated(subset=['buildingid', 'type'], keep='first')]
print(dupes.head(20))

      registrationid  buildingid    boro    zip            type  \
2434          911709      915791  QUEENS  11364  CorporateOwner   
2435          911709      915791  QUEENS  11364  CorporateOwner   
2436          911709      915791  QUEENS  11364  CorporateOwner   
2437          911709      915791  QUEENS  11364  CorporateOwner   
2438          911709      915791  QUEENS  11364  CorporateOwner   
2439          911709      915791  QUEENS  11364  CorporateOwner   
2440          911709      915791  QUEENS  11364  CorporateOwner   
2441          911709      915791  QUEENS  11364  CorporateOwner   
2442          911709      915791  QUEENS  11364  CorporateOwner   
2443          911709      915791  QUEENS  11364  CorporateOwner   
2444          911709      915791  QUEENS  11364  CorporateOwner   
2445          911709      915791  QUEENS  11364  CorporateOwner   
2446          911709      915791  QUEENS  11364  CorporateOwner   
2447          911709      915791  QUEENS  11364  CorporateOwne

In [ ]:
print(df.duplicated(subset=['buildingid', 'type']).sum())

In [ ]:
df = df.drop_duplicates(subset=['buildingid', 'type'])
print(f"Rows after cleaning: {len(df)}")

In [ ]:
df.head(15)

## Section 5: Corporation & Type Analysis

Explore how corporations are distributed across:
- **Contact types** — top corporations per role (CorporateOwner, Agent, Lessee)
- **Zip codes** — number of unique corporations per zip and type
- **Boroughs** — aggregated view of corporate presence across NYC boroughs

In [ ]:
types = df['type'].unique()

for t in types:
    subset = df[df['type'] == t]['corporationname'].value_counts().reset_index()
    subset.columns = ['Corporation Name', 'Count']
    subset.index = subset.index + 1  
    
    print(f"\n{t}")
    display(subset)

In [ ]:
summary = df.groupby(['zip', 'type'])['corporationname'].nunique().unstack(fill_value=0)
display(summary)

In [ ]:
summary = df.groupby(['boro', 'type'])['corporationname'].nunique().unstack(fill_value=0)
display(summary)

## Section 6: Income Data Integration

Load NYC median household income by zip code, classify each zip into income bands relative to the city-wide median, then merge with the building dataset to enable income-based analysis.

In [ ]:
df_income=pd.read_csv(r"C:\Users\Diogo Forte\OneDrive\DTU\4TH SEMESTER\Social data analysis\final_project\sdata_project\data\nyc_zipcode_income.csv")
print(df_income.isna().sum())
df_income = df_income.dropna()
df_income.head()

In [ ]:
median_income=df_income["median_income"]
print(median_income.describe())

median_income = np.array(median_income)

### 6.1 Income Categorization

Classify each zip code into one of four income bands based on its distance from the city median:

| Category | Range |
|---|---|
| **Low** | < 80% of median (< ~$72K) |
| **Low-Middle** | 80–100% (~$72K–$90K) |
| **Middle-High** | 100–120% (~$90K–$108K) |
| **High** | > 120% (> ~$108K) |

In [ ]:
city_median = df_income["median_income"].median()

# income thats 0.8 of median is considered low
low_thr = 0.8 * city_median
mid_thr = 1.0 * city_median
high_thr = 1.2 * city_median

bins = [0, low_thr, mid_thr, high_thr, float("inf")]
labels = ["Low", "Low-Middle", "Middle-High", "High"]

df_income["income_cat"] = pd.cut(
    df_income["median_income"],
    bins=bins,
    labels=labels,
    right=False
)

In [ ]:
print(f"City median income: ${city_median:,.0f}")
print(f"Low threshold: ${low_thr:,.0f}")
print(f"Mid threshold: ${mid_thr:,.0f}")
print(f"High threshold: ${high_thr:,.0f}")

print("\nCategory counts:")
print(df_income["income_cat"].value_counts())

In [ ]:
print(df_income.columns.tolist())

In [ ]:
df_income.rename(columns={"zipcode": "zip","borough":"boro"}, inplace=True)

In [ ]:
df_income = pd.DataFrame(df_income)
df_income.head()

### 6.2 Merging Building Data with Income Data

Merge the income-categorized zip code table with the cleaned building registrations on the `zip` column (inner join — keeps only zip codes present in both datasets).

In [ ]:
df_final = pd.merge(df, df_income, on=["zip"], how="inner")

In [ ]:
df_final = pd.DataFrame(df_final)
df_final.head()

In [ ]:
df_final = df_final.drop(columns=["location","median_income","margin_of_error","boro_y", "businessstreetname", "businesshousenumber"])
df_final.head()

## Section 7: Income-Based Corporation Analysis

Analyse which corporations dominate building ownership across different income areas.  
For each income category, identify the top 10 corporations by number of unique buildings managed.

In [ ]:
summary = df_final.groupby(['income_cat', 'type'])['corporationname'].nunique().unstack(fill_value=0)
display(summary)

In [ ]:
for cat in df_final['income_cat'].cat.categories:
    subset = (
        df_final[df_final['income_cat'] == cat]
        .groupby('corporationname')['buildingid']
        .nunique()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )
    subset.columns = ['Corporation Name', 'Building Count']
    subset.index = subset.index + 1
    print(f"\n--- {cat} Income Areas ---")
    display(subset)


## Section 9: Cross-Borough Ownership & Network Analysis

Identify corporations that own buildings in **more than one borough**, then visualise the inter-borough relationships as a weighted network:
- **Nodes** = NYC boroughs
- **Edges** = two boroughs are connected if at least one corporation owns buildings in both
- **Edge weight** = number of corporations bridging that pair of boroughs

In [ ]:
### 9.1 Corporations Operating Across Multiple Boroughs

corp_boro_count = (
    df_final[df_final['type'] == 'CorporateOwner']
    .groupby('corporationname')['boro_x']
    .nunique()
    .sort_values(ascending=False)
)

multi_boro = corp_boro_count[corp_boro_count > 1]
print(f"Corporations in 2+ boroughs: {len(multi_boro):,}")
print(f"That is {len(multi_boro) / len(corp_boro_count) * 100:.1f}% of all corporate owners\n")

dist = corp_boro_count.value_counts().sort_index()
fig = px.bar(
    x=dist.index.astype(str),
    y=dist.values,
    labels={'x': 'Number of Boroughs', 'y': 'Number of Corporations'},
    title='How Many Boroughs Does Each Corporation Operate In?',
    color=dist.values,
    color_continuous_scale='Blues',
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

# Top 20 multi-borough corporations
top_multi = (
    df_final[
        (df_final['type'] == 'CorporateOwner') &
        (df_final['corporationname'].isin(multi_boro.index))
    ]
    .groupby('corporationname').agg(
        boroughs=('boro_x', lambda x: ', '.join(sorted(x.unique()))),
        borough_count=('boro_x', 'nunique'),
        building_count=('buildingid', 'nunique'),
    )
    .sort_values('building_count', ascending=False)
    .head(20)
    .reset_index()
)
top_multi.index = top_multi.index + 1
display(top_multi)

In [ ]:
### 9.2 Borough Co-Ownership Network

import networkx as nx
from itertools import combinations

corp_to_boros = (
    df_final[df_final['type'] == 'CorporateOwner']
    .groupby('corporationname')['boro_x']
    .apply(set)
)

G = nx.Graph()
G.add_nodes_from(df_final['boro_x'].unique())

for boros in corp_to_boros:
    for b1, b2 in combinations(sorted(boros), 2):
        if G.has_edge(b1, b2):
            G[b1][b2]['weight'] += 1
        else:
            G.add_edge(b1, b2, weight=1)

pos = nx.spring_layout(G, seed=42)
weights = [G[u][v]['weight'] for u, v in G.edges()]
max_w = max(weights)

fig, ax = plt.subplots(figsize=(10, 7))
nx.draw_networkx_nodes(G, pos, node_size=2500, node_color='steelblue', ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_color='white', font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, width=[w / max_w * 10 for w in weights], alpha=0.5, edge_color='gray', ax=ax)
nx.draw_networkx_edge_labels(
    G, pos,
    edge_labels={(u, v): f"{G[u][v]['weight']:,}" for u, v in G.edges()},
    font_size=8, ax=ax
)
ax.set_title('Borough Co-Ownership Network\n(edge weight = corporations owning buildings in both boroughs)', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

## Section 10: Violations & Complaints Analysis

Join HPD violation and 311 complaint data (2020–2025) with the building dataset on `zip` to explore:
- Do **low-income areas** have more violations?
- Which **violation classes** dominate in each income band?
- Which **corporations** operate in zip codes with the highest violation counts?
- **Choropleth map** of violations across NYC zip codes

Violation classes:
- **A** — Non-hazardous
- **B** — Hazardous
- **C** — Immediately hazardous (most severe)

In [ ]:
### 10.1 Load & Aggregate Violations and Complaints by Zip

df_viol = pd.read_csv(r"C:\Users\Diogo Forte\OneDrive\DTU\4TH SEMESTER\Social data analysis\final_project\sdata_project\data\violations20_25_original.csv")
df_comp = pd.read_csv(r"C:\Users\Diogo Forte\OneDrive\DTU\4TH SEMESTER\Social data analysis\final_project\sdata_project\data\complaints20_25_original.csv")

viol_zip = (
    df_viol.groupby('zip').agg(
        total_violations=('violationid', 'count'),
        open_violations=('violationstatus', lambda x: (x == 'Open').sum()),
        class_a=('class', lambda x: (x == 'A').sum()),
        class_b=('class', lambda x: (x == 'B').sum()),
        class_c=('class', lambda x: (x == 'C').sum()),
    )
    .reset_index()
)

comp_zip = (
    df_comp.groupby('incident_zip')
    .size()
    .reset_index(name='complaint_count')
    .rename(columns={'incident_zip': 'zip'})
)

viol_zip['zip'] = viol_zip['zip'].astype(str)
comp_zip['zip'] = comp_zip['zip'].astype(str)

print(f"Violations: {len(df_viol):,} rows across {len(viol_zip)} zip codes")
print(f"Complaints: {len(df_comp):,} rows across {len(comp_zip)} zip codes")
viol_zip.head()

In [ ]:
### 10.2 Violations by Income Category

df_income_copy = df_income[['zip', 'income_cat']].copy()
df_income_copy['zip'] = df_income_copy['zip'].astype(str)

df_viol_income = (
    df_income_copy
    .merge(viol_zip, on='zip', how='left')
    .merge(comp_zip, on='zip', how='left')
)

order = ['Low', 'Low-Middle', 'Middle-High', 'High']

viol_by_income = (
    df_viol_income.groupby('income_cat')[['total_violations', 'open_violations', 'class_c', 'complaint_count']]
    .sum()
    .reindex(order)
    .reset_index()
)

fig = px.bar(
    viol_by_income.melt(id_vars='income_cat', value_vars=['total_violations', 'open_violations', 'class_c']),
    x='income_cat',
    y='value',
    color='variable',
    barmode='group',
    labels={'income_cat': 'Income Category', 'value': 'Count', 'variable': 'Metric'},
    title='HPD Violations by Income Category (2020–2025)',
    color_discrete_map={
        'total_violations': 'steelblue',
        'open_violations': 'orange',
        'class_c': 'red',
    },
    category_orders={'income_cat': order},
)
fig.show()

# Also show complaint count separately
fig2 = px.bar(
    viol_by_income,
    x='income_cat',
    y='complaint_count',
    labels={'income_cat': 'Income Category', 'complaint_count': 'Total Complaints'},
    title='311 Complaints by Income Category (2020–2025)',
    color='complaint_count',
    color_continuous_scale='Oranges',
    category_orders={'income_cat': order},
)
fig2.update_layout(coloraxis_showscale=False)
fig2.show()

In [ ]:
### 10.3 Top Corporations by Violations in Their Operating Zip Codes

viol_lookup = viol_zip.set_index('zip')[['total_violations', 'open_violations', 'class_c']].to_dict(orient='index')

corp_zips = (
    df_final[df_final['type'] == 'CorporateOwner']
    .groupby('corporationname')['zip']
    .apply(lambda x: set(x.astype(str).unique()))
)

rows = []
for corp, zips in corp_zips.items():
    total = open_v = class_c = 0
    for z in zips:
        if z in viol_lookup:
            total   += viol_lookup[z]['total_violations']
            open_v  += viol_lookup[z]['open_violations']
            class_c += viol_lookup[z]['class_c']
    rows.append({'corporationname': corp, 'total_violations': total, 'open_violations': open_v, 'class_c': class_c})

corp_viol_df = pd.DataFrame(rows).sort_values('total_violations', ascending=False)
top15 = corp_viol_df.head(15).reset_index(drop=True)
top15.index += 1
display(top15)

fig = px.bar(
    top15,
    x='corporationname',
    y=['total_violations', 'open_violations', 'class_c'],
    barmode='group',
    labels={'corporationname': 'Corporation', 'value': 'Violations', 'variable': 'Type'},
    title='Top 15 Corporations by Violations in Their Operating Zip Codes',
    color_discrete_map={
        'total_violations': 'steelblue',
        'open_violations': 'orange',
        'class_c': 'red',
    },
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
### 10.4 Choropleth Map — Violations by Zip Code

import json

with open(r"C:\Users\Diogo Forte\OneDrive\DTU\4TH SEMESTER\Social data analysis\final_project\sdata_project\data\geojson\nyc-zip-code-tabulation-areas-polygons.geojson") as f:
    geojson = json.load(f)

fig = px.choropleth_mapbox(
    viol_zip,
    geojson=geojson,
    locations='zip',
    featureidkey='properties.postalCode',
    color='total_violations',
    color_continuous_scale='Reds',
    mapbox_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.75,
    title='Total HPD Violations by Zip Code (2020–2025)',
    labels={'total_violations': 'Total Violations'},
    hover_data={'zip': True, 'total_violations': True, 'open_violations': True, 'class_c': True},
)
fig.show()

# Class C (immediately hazardous) map
fig2 = px.choropleth_mapbox(
    viol_zip,
    geojson=geojson,
    locations='zip',
    featureidkey='properties.postalCode',
    color='class_c',
    color_continuous_scale='YlOrRd',
    mapbox_style='carto-positron',
    center={'lat': 40.7128, 'lon': -74.0060},
    zoom=9,
    opacity=0.75,
    title='Class C (Immediately Hazardous) Violations by Zip Code (2020–2025)',
    labels={'class_c': 'Class C Violations'},
)
fig2.show()